# Pipeline Logging & Monitoring
**Chicago Crime Dataset – ETL Workflow Automation**

Called by ADF at the END of every pipeline run (success or failure).

Responsibilities:
1. Write a structured log record to chicago_crime.pipeline_run_log
2. Compute SLA metrics (duration breach, row anomaly detection)
3. Send an alert email via Azure Monitor Action Group webhook (optional)
4. Display a monitoring summary for the Databricks job output

In [0]:
# ─────────────────────────────────────────────
# 1. PARAMETERS  (all injected by ADF)
# ─────────────────────────────────────────────
dbutils.widgets.text("pipeline_run_id",         "manual",                 "ADF Pipeline Run ID")
dbutils.widgets.text("pipeline_name",           "chicago_crime_etl",      "ADF Pipeline Name")
dbutils.widgets.text("run_date",                "",                       "Run Date (YYYY-MM-DD)")
dbutils.widgets.text("trigger_type",            "Manual",                 "Trigger type (Schedule/Manual/Tumbling)")

# Notebook exit values forwarded by ADF Set Variable activities
dbutils.widgets.text("bronze_status",           "UNKNOWN",                "Bronze exit status")
dbutils.widgets.text("bronze_rows",             "0",                      "Bronze rows written")
dbutils.widgets.text("bronze_duration_sec",     "0",                      "Bronze duration (s)")

dbutils.widgets.text("silver_status",           "UNKNOWN",                "Silver exit status")
dbutils.widgets.text("silver_rows",             "0",                      "Silver rows written")
dbutils.widgets.text("silver_duration_sec",     "0",                      "Silver duration (s)")

dbutils.widgets.text("gold_status",             "UNKNOWN",                "Gold exit status")
dbutils.widgets.text("gold_rows",               "0",                      "Gold rows written")
dbutils.widgets.text("gold_duration_sec",       "0",                      "Gold duration (s)")

dbutils.widgets.text("overall_status",          "SUCCESS",                "Overall pipeline status")
dbutils.widgets.text("failure_reason",          "",                       "Failure reason if any")

# Monitoring config
dbutils.widgets.text("log_container",           "logs",                   "ADLS Log Container")
dbutils.widgets.text("log_path",               "chicago_crime/pipeline/", "Log Path")
dbutils.widgets.text("storage_account",         "etlworkflowautomation", "ADLS Storage Account")
dbutils.widgets.text("alert_webhook_url",       "",                       "Azure Monitor Webhook URL (optional)")
dbutils.widgets.text("sla_max_duration_sec",    "3600",                   "SLA: max acceptable total duration")
dbutils.widgets.text("sla_min_rows",            "1000",                   "SLA: min expected Bronze rows")

PIPELINE_RUN_ID      = dbutils.widgets.get("pipeline_run_id")
PIPELINE_NAME        = dbutils.widgets.get("pipeline_name")
RUN_DATE             = dbutils.widgets.get("run_date")
TRIGGER_TYPE         = dbutils.widgets.get("trigger_type")

BRONZE_STATUS        = dbutils.widgets.get("bronze_status")
BRONZE_ROWS          = int(dbutils.widgets.get("bronze_rows")   or 0)
BRONZE_DUR           = float(dbutils.widgets.get("bronze_duration_sec") or 0)

SILVER_STATUS        = dbutils.widgets.get("silver_status")
SILVER_ROWS          = int(dbutils.widgets.get("silver_rows")   or 0)
SILVER_DUR           = float(dbutils.widgets.get("silver_duration_sec") or 0)

GOLD_STATUS          = dbutils.widgets.get("gold_status")
GOLD_ROWS            = int(dbutils.widgets.get("gold_rows")     or 0)
GOLD_DUR             = float(dbutils.widgets.get("gold_duration_sec") or 0)

OVERALL_STATUS       = dbutils.widgets.get("overall_status")
FAILURE_REASON       = dbutils.widgets.get("failure_reason")

LOG_CONTAINER        = dbutils.widgets.get("log_container")
LOG_PATH             = dbutils.widgets.get("log_path")
STORAGE_ACCOUNT      = dbutils.widgets.get("storage_account")
ALERT_WEBHOOK_URL    = dbutils.widgets.get("alert_webhook_url")
SLA_MAX_DUR          = float(dbutils.widgets.get("sla_max_duration_sec"))
SLA_MIN_ROWS         = int(dbutils.widgets.get("sla_min_rows"))

In [0]:
# ─────────────────────────────────────────────
# 2. IMPORTS
# ─────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType,
    TimestampType, BooleanType
)
from delta.tables import DeltaTable
from datetime import datetime, date
import json, requests, traceback

spark = SparkSession.builder.appName("Pipeline_Logging").getOrCreate()

SECRET_SCOPE = "etlworkflowautomation" 
SECRET_KEY   = "*************************************************"  
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SECRET_KEY
)

RUN_DATE   = RUN_DATE if RUN_DATE else str(date.today())
LOG_TIME   = datetime.utcnow()
TOTAL_DUR  = BRONZE_DUR + SILVER_DUR + GOLD_DUR

In [0]:
# ─────────────────────────────────────────────
# 3. SLA EVALUATION
# ─────────────────────────────────────────────
sla_duration_breach = TOTAL_DUR > SLA_MAX_DUR
sla_row_anomaly     = BRONZE_ROWS < SLA_MIN_ROWS and OVERALL_STATUS == "SUCCESS"
sla_passed          = not sla_duration_breach and not sla_row_anomaly

print("── SLA Evaluation ───────────────────────")
print(f"  Total duration  : {TOTAL_DUR:.1f}s  (max: {SLA_MAX_DUR}s)  {'BREACH' if sla_duration_breach else 'Sucessed'}")
print(f"  Bronze rows     : {BRONZE_ROWS:,}    (min: {SLA_MIN_ROWS})   {'ANOMALY' if sla_row_anomaly else 'Sucessed'}")
print(f"  SLA status      : {'PASSED ' if sla_passed else 'FAILED '}")
print("─────────────────────────────────────────")

── SLA Evaluation ───────────────────────
 Total duration : 0.0s (max: 3600.0s) Sucessed
 Bronze rows : 0 (min: 1000) ANOMALY
 SLA status : FAILED 
─────────────────────────────────────────

In [0]:
# ─────────────────────────────────────────────
# 4. BUILD LOG RECORD
# ─────────────────────────────────────────────
log_record = [
    (
        PIPELINE_RUN_ID,
        PIPELINE_NAME,
        RUN_DATE,
        TRIGGER_TYPE,
        LOG_TIME,

        BRONZE_STATUS,
        BRONZE_ROWS,
        BRONZE_DUR,

        SILVER_STATUS,
        SILVER_ROWS,
        SILVER_DUR,

        GOLD_STATUS,
        GOLD_ROWS,
        GOLD_DUR,

        OVERALL_STATUS,
        TOTAL_DUR,
        FAILURE_REASON,
        sla_passed,
        sla_duration_breach,
        sla_row_anomaly,
    )
]

LOG_SCHEMA = StructType([
    StructField("pipeline_run_id",      StringType(),    False),
    StructField("pipeline_name",        StringType(),    True),
    StructField("run_date",             StringType(),    True),
    StructField("trigger_type",         StringType(),    True),
    StructField("logged_at",            TimestampType(), True),

    StructField("bronze_status",        StringType(),    True),
    StructField("bronze_rows",          LongType(),      True),
    StructField("bronze_duration_sec",  DoubleType(),    True),

    StructField("silver_status",        StringType(),    True),
    StructField("silver_rows",          LongType(),      True),
    StructField("silver_duration_sec",  DoubleType(),    True),

    StructField("gold_status",          StringType(),    True),
    StructField("gold_rows",            LongType(),      True),
    StructField("gold_duration_sec",    DoubleType(),    True),

    StructField("overall_status",       StringType(),    True),
    StructField("total_duration_sec",   DoubleType(),    True),
    StructField("failure_reason",       StringType(),    True),
    StructField("sla_passed",           BooleanType(),   True),
    StructField("sla_duration_breach",  BooleanType(),   True),
    StructField("sla_row_anomaly",      BooleanType(),   True),
])

df_log = spark.createDataFrame(log_record, schema=LOG_SCHEMA)
df_log.show(truncate=False)

+---------------+-----------------+----------+------------+--------------------------+-------------+-----------+-------------------+-------------+-----------+-------------------+-----------+---------+-----------------+--------------+------------------+--------------+----------+-------------------+---------------+
pipeline_run_id|pipeline_name |run_date |trigger_type|logged_at |bronze_status|bronze_rows|bronze_duration_sec|silver_status|silver_rows|silver_duration_sec|gold_status|gold_rows|gold_duration_sec|overall_status|total_duration_sec|failure_reason|sla_passed|sla_duration_breach|sla_row_anomaly|
+---------------+-----------------+----------+------------+--------------------------+-------------+-----------+-------------------+-------------+-----------+-------------------+-----------+---------+-----------------+--------------+------------------+--------------+----------+-------------------+---------------+
manual |chicago_crime_etl|2026-05-20|Manual |2026-05-20 10:24:48.904037|UNKNOWN |0 |0.0 |UNKNOWN |0 |0.0 |UNKNOWN |0 |0.0 |SUCCESS |0.0 | |false |false |true |
+---------------+-----------------+----------+------------+--------------------------+-------------+-----------+-------------------+-------------+-----------+-------------------+-----------+---------+-----------------+--------------+------------------+--------------+----------+-------------------+---------------+

In [0]:
# ─────────────────────────────────────────────
# 5. WRITE LOG TO DELTA TABLE
# ─────────────────────────────────────────────
log_uri = f"wasbs://{LOG_CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/{LOG_PATH}"

try:
    df_log.write.format("delta").mode("append").partitionBy("run_date").save(log_uri)

    spark.sql("CREATE DATABASE IF NOT EXISTS chicago_crime")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS chicago_crime.pipeline_run_log
        USING DELTA
        LOCATION '{log_uri}'
    """)
    print(f"Log record written to chicago_crime.pipeline_run_log")
except Exception as e:
    print(f"Log write failed (non-blocking): {e}")

Log write failed (non-blocking): External tables don't support the wasbs scheme.

In [0]:
# ─────────────────────────────────────────────
# 6. ALERT VIA AZURE MONITOR WEBHOOK (optional)
# ─────────────────────────────────────────────
SHOULD_ALERT = (OVERALL_STATUS != "SUCCESS") or sla_duration_breach or sla_row_anomaly

if ALERT_WEBHOOK_URL and SHOULD_ALERT:
    alert_payload = {
        "text": (
            f"[{OVERALL_STATUS}] {PIPELINE_NAME} | {RUN_DATE}\n"
            f"Run ID: {PIPELINE_RUN_ID}\n"
            f"Bronze: {BRONZE_STATUS} ({BRONZE_ROWS:,} rows, {BRONZE_DUR:.1f}s)\n"
            f"Silver: {SILVER_STATUS} ({SILVER_ROWS:,} rows, {SILVER_DUR:.1f}s)\n"
            f"Gold  : {GOLD_STATUS}   ({GOLD_ROWS:,}  rows, {GOLD_DUR:.1f}s)\n"
            f"SLA   : {'PASSED' if sla_passed else 'FAILED'}  |  "
            f"Duration breach: {sla_duration_breach}  |  Row anomaly: {sla_row_anomaly}\n"
            f"Failure reason: {FAILURE_REASON or 'N/A'}"
        )
    }
    try:
        resp = requests.post(
            ALERT_WEBHOOK_URL,
            data=json.dumps(alert_payload),
            headers={"Content-Type": "application/json"},
            timeout=10
        )
        print(f"Alert sent (HTTP {resp.status_code})")
    except Exception as e:
        print(f"Alert failed (non-blocking): {e}")
else:
    print("No alert required.")

No alert required.

In [0]:
# ─────────────────────────────────────────────
# 7. MONITORING SUMMARY QUERY
# ─────────────────────────────────────────────
print("\n── Last 10 Pipeline Runs ─────────────────")
try:
    spark.sql("""
        SELECT
            run_date,
            pipeline_run_id,
            trigger_type,
            overall_status,
            bronze_rows,
            silver_rows,
            gold_rows,
            ROUND(total_duration_sec, 1) AS total_duration_sec,
            sla_passed,
            logged_at
        FROM chicago_crime.pipeline_run_log
        ORDER BY logged_at DESC
        LIMIT 10
    """).show(truncate=False)
except Exception:
    print("(Log table not yet queryable — first run)")

── Last 10 Pipeline Runs ─────────────────
(Log table not yet queryable — first run)

In [0]:
# ─────────────────────────────────────────────
# 8. DELTA TABLE OPTIMISATION (weekly)
#    Run OPTIMIZE + VACUUM on the log table
#    only on Sundays to avoid overhead.
# ─────────────────────────────────────────────
from datetime import datetime as dt
if dt.now().weekday() == 6:   # Sunday
    print("Running weekly OPTIMIZE & VACUUM on log table...")
    spark.sql("OPTIMIZE chicago_crime.pipeline_run_log ZORDER BY (run_date)")
    spark.sql("VACUUM  chicago_crime.pipeline_run_log RETAIN 168 HOURS")  # 7 days
    print("Optimize & Vacuum done.")

In [0]:
# ─────────────────────────────────────────────
# 9. EXIT
# ─────────────────────────────────────────────
print(f"\nLogging notebook complete. Overall status: {OVERALL_STATUS}")
dbutils.notebook.exit(f"{OVERALL_STATUS}|sla={'PASSED' if sla_passed else 'FAILED'}")